# Top-k Is the Wrong Shape

**Topic:** Decoding strategies &nbsp;·&nbsp; **Difficulty:** Medium

Top-k keeps a fixed number of candidates whether the model is certain or not.
After "the capital of France is" the distribution has one real answer and top-k
50 dutifully keeps 49 wrong ones. After "he opened the" it has hundreds of
reasonable continuations and top-k 50 throws most of them away.

Nucleus sampling keeps the smallest set whose probability mass exceeds `p`, so
the set size adapts. Implement both and measure how the kept-set size responds to
the model's own confidence.

## The data (created in the next cell)

- `logits` `(6, 5000)`: six next-token distributions over a 5,000-word
  vocabulary, ranging from very peaked to nearly uniform.
- `k = 50`, `p = 0.9`, temperatures `[0.7, 1.0, 1.3]`.

## Your task

With **NumPy only**:

1. `softmax_T(logits, T)`: temperature-scaled softmax.
2. `topk_mass(probs, k)`: for each row, the total probability kept by the top
   `k` tokens.
3. `nucleus_size(probs, p)`: for each row, the number of tokens in the smallest
   set whose cumulative probability first reaches `p`.
4. `entropy_bits(probs)`: per row.
5. Set `answers` to a dict with `"entropy"` `(6,)`, `"topk_mass"` `(6,)`,
   `"nucleus_size"` `(6,)` (at `T = 1.0`), and `"nucleus_size_by_T"` `(3, 6)`,
   nucleus sizes at each temperature.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks nucleus size grows with entropy and with temperature, while top-k mass
moves the other way.

In [ ]:
import numpy as np

rng = np.random.default_rng(4001)
V = 5000
sharpness = np.array([12.0, 6.0, 3.0, 1.5, 0.6, 0.15])
base = rng.normal(0, 1, (6, V))
logits = base * sharpness[:, None]
k, p = 50, 0.9
temperatures = [0.7, 1.0, 1.3]
print(f"logits {logits.shape}, six distributions from very peaked to flat")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("entropy", [6], 46.48587796661626, 34.52683932983133, 1e-06),
    ("topk_mass", [6], 3.0099977265936624, -0.8422225909752753, 1e-06),
    ("nucleus_size", [6], 10386.0, 11612.957563931222, 1e-06),
    ("nucleus_size_by_T", [3, 6], 30855.0, -2245.726159593072, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if all((_k in answers) for _k in ("nucleus_size", "entropy", "topk_mass", "nucleus_size_by_T")):
    _e = np.asarray(answers["entropy"], dtype=float)
    _n = np.asarray(answers["nucleus_size"], dtype=float)
    _m = np.asarray(answers["topk_mass"], dtype=float)
    _byT = np.asarray(answers["nucleus_size_by_T"], dtype=float)
    _grows = bool(np.all(np.diff(_n) >= 0)) and bool(np.all(np.diff(_e) >= 0))
    _mass = bool(np.all(np.diff(_m) <= 0))
    _temp = bool(np.all(_byT[2] >= _byT[0]))
    print(f"  {'nucleus tracks entropy':<24} {'ok' if _grows else 'wrong'}")
    print(f"  {'top-k mass falls':<24} {'ok' if _mass else 'wrong'}")
    print(f"  {'hotter -> wider nucleus':<24} {'ok' if _temp else 'wrong'}")
    _ok = _ok and _grows and _mass and _temp

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Entropy is the honest summary of how uncertain the model is, and nucleus size
tracks it almost monotonically. That is the whole design goal. Top-k's kept mass
does the opposite: on a peaked distribution the top 50 capture essentially
everything (so `k` is far too generous), and on a flat one they capture a small
fraction (so `k` is far too strict). Raising the temperature flattens the
distribution, which grows the nucleus and shrinks top-k's mass, and is why the
two knobs interact and should not be tuned independently.

```python
import numpy as np

def softmax_T(z, T):
    s = z / T
    s = s - s.max(axis=-1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=-1, keepdims=True)

def topk_mass(P, k):
    srt = np.sort(P, axis=-1)[:, ::-1]
    return srt[:, :k].sum(axis=-1)

def nucleus_size(P, p):
    srt = np.sort(P, axis=-1)[:, ::-1]
    cum = np.cumsum(srt, axis=-1)
    return (cum < p).sum(axis=-1) + 1

def entropy_bits(P):
    Q = np.where(P > 0, P, 1.0)
    return -(P * np.log2(Q)).sum(axis=-1)

P1 = softmax_T(logits, 1.0)

sizes_by_T = np.stack([nucleus_size(softmax_T(logits, T), p).astype(float)
                       for T in temperatures])

answers = {
    "entropy": entropy_bits(P1),
    "topk_mass": topk_mass(P1, k),
    "nucleus_size": nucleus_size(P1, p).astype(float),
    "nucleus_size_by_T": sizes_by_T,
}
```